# MarkProbe Core: weight-only fine-tuning

Load a compatible v6 checkpoint, adjust settings, and train on an existing image directory. The source checkpoint is protected from overwrite. Per-epoch metrics, curves, encoded images, and residual samples are saved beside the new checkpoint.

In [ ]:
from pathlib import Path
import json
import sys
import torch
PROJECT = Path.cwd().resolve()
if not (PROJECT / 'markprobe_core').is_dir(): PROJECT = PROJECT.parent
if not (PROJECT / 'markprobe_core').is_dir(): raise FileNotFoundError('Open from MarkProbe-Core or set PROJECT manually.')
sys.path.insert(0, str(PROJECT))
from markprobe_core import DEFAULT_FINETUNE_SETTINGS, run_finetune
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DATA_DIR = Path('/workspace/dataset/coco2014')
SOURCE_CHECKPOINT = PROJECT / 'runs/markprobe_core.last.pt'
OUTPUT_CHECKPOINT = PROJECT / 'runs/markprobe_core_finetune.pt'
LIMIT = None  # Set to 512 for a pipeline smoke test only.
if not DATA_DIR.is_dir(): raise FileNotFoundError(f'Dataset directory not found: {DATA_DIR}')
if not SOURCE_CHECKPOINT.is_file(): raise FileNotFoundError(f'Source checkpoint not found: {SOURCE_CHECKPOINT}')
print(f'Device: {DEVICE}')
print(f'Dataset: {DATA_DIR.resolve()}')
print(f'Source: {SOURCE_CHECKPOINT.resolve()}')
print(f'New output: {OUTPUT_CHECKPOINT.resolve()}')

## Fine-tuning strategy

Edit this dictionary to change supported `train_model` settings. For a future loss or schedule, expose its parameter in `train_model` and add it here; both this notebook and the command-line script pass settings through the same `run_finetune` helper.

In [ ]:
SETTINGS = dict(DEFAULT_FINETUNE_SETTINGS)
SETTINGS.update({
    'epochs': 10,
    'batch_size': 4,
    'lr': 1e-5,
    'dual_branch': True,
    'crop_scale_min': 0.80,
    'crop_scale_max': 0.95,
    'image_weight': 2.0,
    'lpips_weight': 1.5,
    'gan_weight': 0.5,
    'use_gan': True,
    'edge_delay_steps': 0,
    'edge_ramp_steps': 5000,
    'epoch_export_samples': 4,
})
print(json.dumps(SETTINGS, indent=2, default=str))

## Train

The model weights are initialized from the source checkpoint. Optimizer/discriminator state, epoch, global step, ramps, and RNG state are reset for the new run.

In [ ]:
model = run_finetune(
    DATA_DIR,
    SOURCE_CHECKPOINT,
    OUTPUT_CHECKPOINT,
    settings=SETTINGS,
    device=DEVICE,
    limit=LIMIT,
)

## Inspect results

Epoch metrics are recorded as JSONL. Per-epoch inference artifacts include the cover, encoded image, residual visualization, decoded bits, and per-sample metrics.

In [ ]:
metrics_path = OUTPUT_CHECKPOINT.with_name(OUTPUT_CHECKPOINT.stem + '.metrics.jsonl')
records = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines() if line.strip()]
for record in records:
    scenarios = record.get('validation', {}).get('scenarios', {})
    print(f"epoch={record['epoch']} step={record['global_step']}")
    for name in ('clean_png', 'photo', 'crop_photo'):
        metric = scenarios.get(name, {})
        if metric: print(f"  {name}: BER={metric.get('ber')} accuracy={metric.get('bit_accuracy')}")
artifact_dir = OUTPUT_CHECKPOINT.with_name(OUTPUT_CHECKPOINT.stem + '.epochs')
print(f'Artifacts: {artifact_dir.resolve()}')
print(f'Best checkpoint: {OUTPUT_CHECKPOINT.resolve()}')